# Final Analysis — Automated Pipeline

This notebook fills `finalAnalysis.csv` by:
1. Querying each RAG server (Vector, Graph, Hybrid) for answers
2. Collecting sources/triplets used
3. Computing RAGAS metrics (faithfulness, answer_relevancy, context_precision, context_recall)

**Prerequisites:**
- VectorRAG server running on `localhost:8000`
- GraphRAG server running on `localhost:8001`  
- HybridRAG server running on `localhost:8002`
- Document already uploaded (doc_id: `ceb7c087eb8c`)

**Resume-safe:** Every cell saves to CSV after each question and skips already-filled rows on re-run.
If an API key limit is hit mid-way, just change the key and re-run the same cell — it picks up where it left off.

In [18]:
# ============================================================
# Cell 0: Setup & Configuration
# ============================================================

import pandas as pd
import requests
import json
import time
import os
import sys

# --- Config ---
CSV_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'finalAnalysis.csv')
VECTOR_API  = 'http://localhost:8000'
GRAPH_API   = 'http://localhost:8001'
HYBRID_API  = 'http://localhost:8002'
DOCUMENT_ID = 'ceb7c087eb8c'

def is_empty(val):
    """Check if a cell value is empty/NaN/needs filling."""
    if pd.isna(val):
        return True
    s = str(val).strip()
    return s == '' or s == 'nan'

# Load the CSV
df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} questions from finalAnalysis.csv')
print(f'Columns: {list(df.columns)}')
print()
df[['question', 'question_type', 'ground_truth']].head(20)

Loaded 18 questions from finalAnalysis.csv
Columns: ['question', 'question_type', 'ground_truth', 'vectorRAG_answer', 'graphRAG_answer', 'hybridRAG_answer', 'vectorRAG_sources', 'graphRAG_triplets', 'hybridRAG_sources', 'vector_faithfulness', 'graph_faithfulness', 'hybrid_faithfulness', 'vector_answer_relevancy', 'graph_answer_relevancy', 'hybrid_answer_relevancy', 'vector_context_precision', 'graph_context_precision', 'hybrid_context_precision', 'vector_context_recall', 'graph_context_recall', 'hybrid_context_recall', 'combined_faithfulness', 'combined_answer_relevancy', 'combined_context_precision', 'combined_context_recall']



,question,question_type,ground_truth
0,What is the maximum air ambulance travel dista...,Basic Retrieval,The maximum air ambulance travel distance cove...
1,What expenses are included under routine medic...,Basic Retrieval,"Routine medical care includes pharmacy, diagno..."
2,What preventive care services are covered for ...,Basic Retrieval,Preventive care services for a newborn baby in...
3,Is infertility treatment covered under the Wel...,Basic Retrieval,"No, infertility treatments are not covered und..."
4,What is the mode of payment for air ambulance ...,Basic Retrieval,Air ambulance claims are payable only through ...
5,If an insured person travels 300 km using an a...,Numerical Reasoning,If an insured person travels 300 km using an a...
6,What happens if the air ambulance distance exc...,Numerical Reasoning,"If the air ambulance distance exceeds 150 km, ..."
7,Under what conditions will air ambulance servi...,Multi-Clause Retrieval,Air ambulance services will be approved only f...
8,List all exclusions under the Air Ambulance Co...,Multi-Clause Retrieval,Exclusions include transfer between similar me...
9,What are the available coverage periods under ...,Multi-Clause Retrieval,Coverage periods include: from onset of pregna...


In [19]:
# ============================================================
# Cell 1: Fill vectorRAG_answer & vectorRAG_sources
# ============================================================
# ► Skips rows that already have vectorRAG_answer filled.
# ► Saves CSV after EACH question so progress is never lost.

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Filling: vectorRAG_answer & vectorRAG_sources')
print('=' * 60)

skipped = 0
filled  = 0

for idx, row in df.iterrows():
    question = row['question']
    
    # --- SKIP if already filled ---
    if not is_empty(row.get('vectorRAG_answer')):
        skipped += 1
        print(f'[{idx+1}/{len(df)}] SKIP (already filled)')
        continue
    
    print(f'\n[{idx+1}/{len(df)}] {question[:80]}...')
    
    try:
        resp = requests.post(
            f'{VECTOR_API}/api/query',
            json={'document_id': DOCUMENT_ID, 'query': question, 'top_k': 5},
            timeout=120
        )
        resp.raise_for_status()
        data = resp.json()
        
        df.at[idx, 'vectorRAG_answer'] = data.get('answer', '')
        sources = data.get('sources', [])
        df.at[idx, 'vectorRAG_sources'] = json.dumps(sources)
        
        print(f'  ✓ Answer: {data["answer"][:100]}...')
        print(f'  ✓ Sources: {len(sources)} chunks retrieved')
        filled += 1
        
    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        print(f'  ⚠ Stopping here. Change API key if needed and re-run this cell.')
        df.to_csv(CSV_PATH, index=False)
        raise  # Stop execution so user can fix the issue
    
    # --- SAVE after each question ---
    df.to_csv(CSV_PATH, index=False)
    time.sleep(1)  # Rate limiting

print(f'\n{"=" * 60}')
print(f'  ✓ Done! Skipped: {skipped}, Newly filled: {filled}')
print(f'  ✓ Saved to CSV')
print('=' * 60)

  Filling: vectorRAG_answer & vectorRAG_sources
[1/18] SKIP (already filled)
[2/18] SKIP (already filled)
[3/18] SKIP (already filled)
[4/18] SKIP (already filled)
[5/18] SKIP (already filled)
[6/18] SKIP (already filled)
[7/18] SKIP (already filled)
[8/18] SKIP (already filled)
[9/18] SKIP (already filled)
[10/18] SKIP (already filled)
[11/18] SKIP (already filled)
[12/18] SKIP (already filled)
[13/18] SKIP (already filled)
[14/18] SKIP (already filled)
[15/18] SKIP (already filled)
[16/18] SKIP (already filled)
[17/18] SKIP (already filled)
[18/18] SKIP (already filled)

  ✓ Done! Skipped: 18, Newly filled: 0
  ✓ Saved to CSV


In [20]:
# ============================================================
# Cell 2: Fill graphRAG_answer & graphRAG_triplets
# ============================================================
# ► Skips rows that already have graphRAG_answer filled.
# ► Saves CSV after EACH question.

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Filling: graphRAG_answer & graphRAG_triplets')
print('=' * 60)

skipped = 0
filled  = 0

for idx, row in df.iterrows():
    question = row['question']
    
    # --- SKIP if already filled ---
    if not is_empty(row.get('graphRAG_answer')):
        skipped += 1
        print(f'[{idx+1}/{len(df)}] SKIP (already filled)')
        continue
    
    print(f'\n[{idx+1}/{len(df)}] {question[:80]}...')
    
    try:
        resp = requests.post(
            f'{GRAPH_API}/api/graph/query',
            json={'document_id': DOCUMENT_ID, 'query': question},
            timeout=120
        )
        resp.raise_for_status()
        data = resp.json()
        
        df.at[idx, 'graphRAG_answer'] = data.get('answer', '')
        triplets = data.get('triplets_used', [])
        df.at[idx, 'graphRAG_triplets'] = json.dumps(triplets)
        
        print(f'  ✓ Answer: {data["answer"][:100]}...')
        print(f'  ✓ Triplets: {len(triplets)} used')
        filled += 1
        
    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        print(f'  ⚠ Stopping here. Change API key if needed and re-run this cell.')
        df.to_csv(CSV_PATH, index=False)
        raise
    
    # --- SAVE after each question ---
    df.to_csv(CSV_PATH, index=False)
    time.sleep(1)

print(f'\n{"=" * 60}')
print(f'  ✓ Done! Skipped: {skipped}, Newly filled: {filled}')
print(f'  ✓ Saved to CSV')
print('=' * 60)

  Filling: graphRAG_answer & graphRAG_triplets
[1/18] SKIP (already filled)
[2/18] SKIP (already filled)
[3/18] SKIP (already filled)
[4/18] SKIP (already filled)
[5/18] SKIP (already filled)
[6/18] SKIP (already filled)
[7/18] SKIP (already filled)
[8/18] SKIP (already filled)
[9/18] SKIP (already filled)
[10/18] SKIP (already filled)
[11/18] SKIP (already filled)
[12/18] SKIP (already filled)
[13/18] SKIP (already filled)
[14/18] SKIP (already filled)
[15/18] SKIP (already filled)
[16/18] SKIP (already filled)
[17/18] SKIP (already filled)
[18/18] SKIP (already filled)

  ✓ Done! Skipped: 18, Newly filled: 0
  ✓ Saved to CSV


In [21]:
# ============================================================
# Cell 3: Fill hybridRAG_answer & hybridRAG_sources
# ============================================================
# ► Skips rows that already have hybridRAG_answer filled.
# ► Saves CSV after EACH question.
# ► Requires vectorRAG_answer & graphRAG_answer to be filled first.

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Filling: hybridRAG_answer & hybridRAG_sources')
print('=' * 60)

skipped = 0
filled  = 0

for idx, row in df.iterrows():
    question = row['question']
    
    # --- SKIP if already filled ---
    if not is_empty(row.get('hybridRAG_answer')):
        skipped += 1
        print(f'[{idx+1}/{len(df)}] SKIP (already filled)')
        continue
    
    vector_answer = str(row.get('vectorRAG_answer', '') or '')
    graph_answer  = str(row.get('graphRAG_answer', '') or '')
    
    print(f'\n[{idx+1}/{len(df)}] {question[:80]}...')
    
    try:
        resp = requests.post(
            f'{HYBRID_API}/api/hybrid/compose',
            json={
                'query': question,
                'vector_answer': vector_answer,
                'graph_answer': graph_answer,
            },
            timeout=120
        )
        resp.raise_for_status()
        data = resp.json()
        
        df.at[idx, 'hybridRAG_answer'] = data.get('answer', '')
        
        # Hybrid sources = combination of vector sources + graph triplets
        vector_sources_str = str(row.get('vectorRAG_sources', '[]') or '[]')
        graph_triplets_str = str(row.get('graphRAG_triplets', '[]') or '[]')
        
        try:
            v_sources = json.loads(vector_sources_str)
        except:
            v_sources = []
        try:
            g_triplets = json.loads(graph_triplets_str)
        except:
            g_triplets = []
        
        hybrid_sources = {
            'vector_sources': v_sources,
            'graph_triplets': g_triplets
        }
        df.at[idx, 'hybridRAG_sources'] = json.dumps(hybrid_sources)
        
        print(f'  ✓ Answer: {data["answer"][:100]}...')
        filled += 1
        
    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        print(f'  ⚠ Stopping here. Change API key if needed and re-run this cell.')
        df.to_csv(CSV_PATH, index=False)
        raise
    
    # --- SAVE after each question ---
    df.to_csv(CSV_PATH, index=False)
    time.sleep(1)

print(f'\n{"=" * 60}')
print(f'  ✓ Done! Skipped: {skipped}, Newly filled: {filled}')
print(f'  ✓ Saved to CSV')
print('=' * 60)

  Filling: hybridRAG_answer & hybridRAG_sources
[1/18] SKIP (already filled)
[2/18] SKIP (already filled)
[3/18] SKIP (already filled)
[4/18] SKIP (already filled)
[5/18] SKIP (already filled)
[6/18] SKIP (already filled)
[7/18] SKIP (already filled)
[8/18] SKIP (already filled)
[9/18] SKIP (already filled)
[10/18] SKIP (already filled)
[11/18] SKIP (already filled)
[12/18] SKIP (already filled)
[13/18] SKIP (already filled)
[14/18] SKIP (already filled)
[15/18] SKIP (already filled)
[16/18] SKIP (already filled)
[17/18] SKIP (already filled)
[18/18] SKIP (already filled)

  ✓ Done! Skipped: 18, Newly filled: 0
  ✓ Saved to CSV


In [22]:
# ============================================================
# Cell 4: Setup RAGAS evaluation (LLM + Embeddings)
# ============================================================
# This cell configures the LLM and EMBEDDINGS for RAGAS.
# ► answer_relevancy REQUIRES embeddings — without this it returns NaN.
# ► Re-run this cell after changing the API key in .env.

# !pip install ragas langchain-openai langchain-community datasets

import os
from dotenv import load_dotenv

# Load .env from project root (override=True to pick up new keys)
env_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '.env')
load_dotenv(dotenv_path=env_path, override=True)

OPENAI_API_KEY = os.getenv('OPEN_API_KEY') or os.getenv('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print(f'OpenAI API Key loaded: {"Yes" if OPENAI_API_KEY else "No"}')
print(f'Key prefix: {OPENAI_API_KEY[:20]}...' if OPENAI_API_KEY else 'NOT SET')

# --- Setup LLM + Embeddings for RAGAS ---
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(ChatOpenAI(
    model='gpt-4o',
    api_key=OPENAI_API_KEY,
    temperature=0,
))

ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=OPENAI_API_KEY,
))

print('\n✓ RAGAS LLM configured:        gpt-4o')
print('✓ RAGAS Embeddings configured:  text-embedding-3-small')
print('\nTip: If you change the API key in .env, re-run this cell to reload it.')

OpenAI API Key loaded: Yes
Key prefix: sk-proj-Jh16sKomVbMT...


C:\Users\grove\AppData\Local\Temp\ipykernel_18304\1965011540.py:28: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(



✓ RAGAS LLM configured:        gpt-4o
✓ RAGAS Embeddings configured:  text-embedding-3-small

Tip: If you change the API key in .env, re-run this cell to reload it.


C:\Users\grove\AppData\Local\Temp\ipykernel_18304\1965011540.py:34: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(


In [23]:
# ============================================================
# Cell 5: RAGAS helper functions
# ============================================================

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from datasets import Dataset


def batch_ragas_evaluate(questions, answers, contexts_list, ground_truths):
    """
    Batch-evaluate RAGAS metrics for multiple questions at once.
    This is MUCH faster than calling evaluate() per question
    because it parallelizes LLM calls internally.
    
    Returns:
        List of dicts, one per question, with keys:
        faithfulness, answer_relevancy, context_precision, context_recall
    """
    # Sanitize inputs
    clean_contexts = []
    for ctx in contexts_list:
        if not ctx:
            clean_contexts.append(['No context retrieved.'])
        else:
            clean_contexts.append(ctx)
    
    data = {
        'question': questions,
        'answer': answers,
        'contexts': clean_contexts,
        'ground_truth': ground_truths,
    }
    
    dataset = Dataset.from_dict(data)
    
    result = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    
    # Convert result to per-row dicts
    result_df = result.to_pandas()
    per_row = []
    for _, row in result_df.iterrows():
        per_row.append({
            'faithfulness': row.get('faithfulness'),
            'answer_relevancy': row.get('answer_relevancy'),
            'context_precision': row.get('context_precision'),
            'context_recall': row.get('context_recall'),
        })
    return per_row


def extract_context_texts_from_sources(sources_json_str: str) -> list:
    """Extract plain text contexts from vectorRAG sources JSON."""
    try:
        sources = json.loads(str(sources_json_str))
        if isinstance(sources, list):
            return [s.get('text', '') for s in sources if s.get('text')]
    except:
        pass
    return []


def extract_context_texts_from_triplets(triplets_json_str: str) -> list:
    """Convert graphRAG triplets to plain text contexts."""
    try:
        triplets = json.loads(str(triplets_json_str))
        if isinstance(triplets, list):
            texts = []
            for t in triplets:
                subj = t.get('subject', '?')
                pred = t.get('predicate', '?').replace('_', ' ').lower()
                obj  = t.get('object', '?')
                texts.append(f'{subj} {pred} {obj}')
            return texts if texts else []
    except:
        pass
    return []


def extract_context_texts_from_hybrid(hybrid_json_str: str) -> list:
    """Extract combined contexts from hybridRAG sources JSON."""
    try:
        hybrid = json.loads(str(hybrid_json_str))
        contexts = []
        for s in hybrid.get('vector_sources', []):
            if s.get('text'):
                contexts.append(s['text'])
        for t in hybrid.get('graph_triplets', []):
            subj = t.get('subject', '?')
            pred = t.get('predicate', '?').replace('_', ' ').lower()
            obj  = t.get('object', '?')
            contexts.append(f'{subj} {pred} {obj}')
        return contexts if contexts else []
    except:
        pass
    return []


print('✓ RAGAS helper functions defined (batch mode).')

✓ RAGAS helper functions defined (batch mode).


C:\Users\grove\AppData\Local\Temp\ipykernel_18304\3361628844.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\grove\AppData\Local\Temp\ipykernel_18304\3361628844.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\grove\AppData\Local\Temp\ipykernel_18304\3361628844.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\grove\AppData\Local\Temp\ipykernel_18

In [24]:
# ============================================================
# Cell 6: Fill VectorRAG RAGAS metrics (BATCH)
# ============================================================
# Columns: vector_faithfulness, vector_answer_relevancy,
#          vector_context_precision, vector_context_recall
#
# ► Collects all unfilled rows → evaluates in ONE batch call.
# ► Much faster than per-question evaluation.
# ► Saves results to CSV after the batch completes.

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Computing RAGAS metrics for VectorRAG (BATCH)')
print('=' * 60)

# Find unfilled rows
unfilled_indices = []
for idx, row in df.iterrows():
    if is_empty(row.get('vector_faithfulness')):
        unfilled_indices.append(idx)
    else:
        print(f'[{idx+1}/{len(df)}] SKIP (already computed)')

if not unfilled_indices:
    print('\n✓ All rows already computed. Nothing to do.')
else:
    print(f'\n→ {len(unfilled_indices)} questions to evaluate (skipped {len(df) - len(unfilled_indices)})')
    
    # Prepare batch data
    questions = []
    answers = []
    contexts_list = []
    ground_truths = []
    
    for idx in unfilled_indices:
        row = df.iloc[idx]
        questions.append(row['question'])
        answers.append(str(row.get('vectorRAG_answer', '') or ''))
        ground_truths.append(str(row.get('ground_truth', '') or ''))
        contexts_list.append(extract_context_texts_from_sources(
            str(row.get('vectorRAG_sources', '[]') or '[]')
        ))
    
    print(f'→ Running batch RAGAS evaluation...')
    t0 = time.time()
    
    try:
        results = batch_ragas_evaluate(questions, answers, contexts_list, ground_truths)
        
        # Write results back
        for i, idx in enumerate(unfilled_indices):
            df.at[idx, 'vector_faithfulness']      = results[i]['faithfulness']
            df.at[idx, 'vector_answer_relevancy']  = results[i]['answer_relevancy']
            df.at[idx, 'vector_context_precision'] = results[i]['context_precision']
            df.at[idx, 'vector_context_recall']    = results[i]['context_recall']
        
        elapsed = time.time() - t0
        df.to_csv(CSV_PATH, index=False)
        print(f'\n✓ Batch complete! {len(unfilled_indices)} questions evaluated in {elapsed:.1f}s')
        print(f'✓ Saved to CSV')
        
    except Exception as e:
        print(f'\n✗ Batch evaluation failed: {e}')
        print(f'⚠ Change API key in .env → re-run Cell 4 → re-run this cell')
        raise

print('=' * 60)

  Computing RAGAS metrics for VectorRAG (BATCH)
[1/18] SKIP (already computed)
[2/18] SKIP (already computed)
[3/18] SKIP (already computed)
[4/18] SKIP (already computed)
[5/18] SKIP (already computed)
[6/18] SKIP (already computed)
[7/18] SKIP (already computed)
[8/18] SKIP (already computed)
[9/18] SKIP (already computed)
[10/18] SKIP (already computed)
[11/18] SKIP (already computed)
[12/18] SKIP (already computed)
[13/18] SKIP (already computed)
[14/18] SKIP (already computed)
[15/18] SKIP (already computed)
[16/18] SKIP (already computed)
[17/18] SKIP (already computed)
[18/18] SKIP (already computed)

✓ All rows already computed. Nothing to do.


In [25]:
# ============================================================
# Cell 7: Fill GraphRAG RAGAS metrics (BATCH)
# ============================================================
# Columns: graph_faithfulness, graph_answer_relevancy,
#          graph_context_precision, graph_context_recall

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Computing RAGAS metrics for GraphRAG (BATCH)')
print('=' * 60)

# Find unfilled rows
unfilled_indices = []
for idx, row in df.iterrows():
    if is_empty(row.get('graph_faithfulness')):
        unfilled_indices.append(idx)
    else:
        print(f'[{idx+1}/{len(df)}] SKIP (already computed)')

if not unfilled_indices:
    print('\n✓ All rows already computed. Nothing to do.')
else:
    print(f'\n→ {len(unfilled_indices)} questions to evaluate (skipped {len(df) - len(unfilled_indices)})')
    
    questions = []
    answers = []
    contexts_list = []
    ground_truths = []
    
    for idx in unfilled_indices:
        row = df.iloc[idx]
        questions.append(row['question'])
        answers.append(str(row.get('graphRAG_answer', '') or ''))
        ground_truths.append(str(row.get('ground_truth', '') or ''))
        contexts_list.append(extract_context_texts_from_triplets(
            str(row.get('graphRAG_triplets', '[]') or '[]')
        ))
    
    print(f'→ Running batch RAGAS evaluation...')
    t0 = time.time()
    
    try:
        results = batch_ragas_evaluate(questions, answers, contexts_list, ground_truths)
        
        for i, idx in enumerate(unfilled_indices):
            df.at[idx, 'graph_faithfulness']      = results[i]['faithfulness']
            df.at[idx, 'graph_answer_relevancy']  = results[i]['answer_relevancy']
            df.at[idx, 'graph_context_precision'] = results[i]['context_precision']
            df.at[idx, 'graph_context_recall']    = results[i]['context_recall']
        
        elapsed = time.time() - t0
        df.to_csv(CSV_PATH, index=False)
        print(f'\n✓ Batch complete! {len(unfilled_indices)} questions evaluated in {elapsed:.1f}s')
        print(f'✓ Saved to CSV')
        
    except Exception as e:
        print(f'\n✗ Batch evaluation failed: {e}')
        print(f'⚠ Change API key in .env → re-run Cell 4 → re-run this cell')
        raise

print('=' * 60)

  Computing RAGAS metrics for GraphRAG (BATCH)
[1/18] SKIP (already computed)
[2/18] SKIP (already computed)
[3/18] SKIP (already computed)
[4/18] SKIP (already computed)
[5/18] SKIP (already computed)

→ 13 questions to evaluate (skipped 5)
→ Running batch RAGAS evaluation...


Evaluating:   2%|▏         | 1/52 [00:23<20:20, 23.93s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  62%|██████▏   | 32/52 [03:08<05:28, 16.44s/it]Exception raised in Job[2]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  83%|████████▎ | 43/52 [03:58<01:01,  6.87s/it]Exception raised in Job[30]: TimeoutError()
Exception raised in Job[26]: TimeoutError()
Evaluating: 100%|██████████| 52/52 [06:13<00:00,  7.17s/it]



✓ Batch complete! 13 questions evaluated in 383.0s
✓ Saved to CSV


In [26]:
# ============================================================
# Cell 8: Fill HybridRAG RAGAS metrics (BATCH)
# ============================================================
# Columns: hybrid_faithfulness, hybrid_answer_relevancy,
#          hybrid_context_precision, hybrid_context_recall

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Computing RAGAS metrics for HybridRAG (BATCH)')
print('=' * 60)

# Find unfilled rows
unfilled_indices = []
for idx, row in df.iterrows():
    if is_empty(row.get('hybrid_faithfulness')):
        unfilled_indices.append(idx)
    else:
        print(f'[{idx+1}/{len(df)}] SKIP (already computed)')

if not unfilled_indices:
    print('\n✓ All rows already computed. Nothing to do.')
else:
    print(f'\n→ {len(unfilled_indices)} questions to evaluate (skipped {len(df) - len(unfilled_indices)})')
    
    questions = []
    answers = []
    contexts_list = []
    ground_truths = []
    
    for idx in unfilled_indices:
        row = df.iloc[idx]
        questions.append(row['question'])
        answers.append(str(row.get('hybridRAG_answer', '') or ''))
        ground_truths.append(str(row.get('ground_truth', '') or ''))
        contexts_list.append(extract_context_texts_from_hybrid(
            str(row.get('hybridRAG_sources', '{}') or '{}')
        ))
    
    print(f'→ Running batch RAGAS evaluation...')
    t0 = time.time()
    
    try:
        results = batch_ragas_evaluate(questions, answers, contexts_list, ground_truths)
        
        for i, idx in enumerate(unfilled_indices):
            df.at[idx, 'hybrid_faithfulness']      = results[i]['faithfulness']
            df.at[idx, 'hybrid_answer_relevancy']  = results[i]['answer_relevancy']
            df.at[idx, 'hybrid_context_precision'] = results[i]['context_precision']
            df.at[idx, 'hybrid_context_recall']    = results[i]['context_recall']
        
        elapsed = time.time() - t0
        df.to_csv(CSV_PATH, index=False)
        print(f'\n✓ Batch complete! {len(unfilled_indices)} questions evaluated in {elapsed:.1f}s')
        print(f'✓ Saved to CSV')
        
    except Exception as e:
        print(f'\n✗ Batch evaluation failed: {e}')
        print(f'⚠ Change API key in .env → re-run Cell 4 → re-run this cell')
        raise

print('=' * 60)

  Computing RAGAS metrics for HybridRAG (BATCH)

→ 18 questions to evaluate (skipped 0)
→ Running batch RAGAS evaluation...


Evaluating:   1%|▏         | 1/72 [00:23<27:32, 23.27s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  51%|█████▏    | 37/72 [02:48<04:55,  8.44s/it]Exception raised in Job[6]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Evaluating:  61%|██████    | 44/72 [03:28<02:51,  6.12s/it]Exception raised in Job[22]: TimeoutError()
Exception raised in Job[18]: TimeoutError()
Evaluating:  65%|██████▌   | 47/72 [03:38<02:01,  4.85s/it]Exception raised in Job[26]: TimeoutError()
Exception raised in Job[30]: TimeoutError()
Evaluating:  76%|███████▋  | 55/72 [04:05<01:15,  4.43s/it]Exception raised in Job[42]: TimeoutError()
Exception raised in Job[39]: TimeoutError()
Evaluating: 100%|█████


✓ Batch complete! 18 questions evaluated in 434.3s
✓ Saved to CSV


In [27]:
# ============================================================
# Cell 9: Fill combined metrics (averages)
# ============================================================
# Columns: combined_faithfulness, combined_answer_relevancy,
#          combined_context_precision, combined_context_recall

import numpy as np

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Computing Combined (Average) Metrics')
print('=' * 60)

metric_names = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
rag_prefixes = ['vector', 'graph', 'hybrid']

skipped = 0
filled  = 0

for idx, row in df.iterrows():
    question = row['question']
    
    # --- SKIP if already computed ---
    if not is_empty(row.get('combined_faithfulness')):
        skipped += 1
        print(f'[{idx+1}/{len(df)}] SKIP (already computed)')
        continue
    
    print(f'[{idx+1}/{len(df)}] {question[:60]}...')
    
    for metric in metric_names:
        values = []
        for prefix in rag_prefixes:
            col = f'{prefix}_{metric}'
            val = row.get(col)
            if pd.notna(val):
                try:
                    values.append(float(val))
                except (ValueError, TypeError):
                    pass
        
        if values:
            df.at[idx, f'combined_{metric}'] = round(np.mean(values), 4)
        else:
            df.at[idx, f'combined_{metric}'] = None
    
    filled += 1

# Save
df.to_csv(CSV_PATH, index=False)
print(f'\n{"=" * 60}')
print(f'  ✓ Done! Skipped: {skipped}, Newly computed: {filled}')
print('=' * 60)

  Computing Combined (Average) Metrics
[1/18] What is the maximum air ambulance travel distance covered un...
[2/18] What expenses are included under routine medical care for We...
[3/18] What preventive care services are covered for a newborn baby...
[4/18] Is infertility treatment covered under the Well Mother cover...
[5/18] What is the mode of payment for air ambulance claims?...
[6/18] If an insured person travels 300 km using an air ambulance, ...
[7/18] What happens if the air ambulance distance exceeds 150 km?...
[8/18] Under what conditions will air ambulance services be approve...
[9/18] List all exclusions under the Air Ambulance Cover....
[10/18] What are the available coverage periods under the Well Mothe...
[11/18] Does the policy cover transportation from hospital to home a...
[12/18] Does the policy cover air ambulance expenses outside India?...
[13/18] Will the insurer pay if treatment is available at the curren...
[14/18] Are unlawful acts covered under air ambulance 

In [28]:
# ============================================================
# Cell 10: Final Summary & Verification
# ============================================================

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  FINAL ANALYSIS SUMMARY')
print('=' * 60)
print(f'\nTotal questions: {len(df)}')
print(f'Columns: {len(df.columns)}')
print()

# Check completeness
for col in df.columns:
    non_null = df[col].notna().sum()
    pct = (non_null / len(df)) * 100
    status = '✓' if pct == 100 else '⚠' if pct > 0 else '✗'
    print(f'  {status} {col}: {non_null}/{len(df)} filled ({pct:.0f}%)')

print()
print('─' * 60)
print('METRIC AVERAGES ACROSS ALL QUESTIONS')
print('─' * 60)

metric_cols = [
    'vector_faithfulness', 'vector_answer_relevancy', 'vector_context_precision', 'vector_context_recall',
    'graph_faithfulness', 'graph_answer_relevancy', 'graph_context_precision', 'graph_context_recall',
    'hybrid_faithfulness', 'hybrid_answer_relevancy', 'hybrid_context_precision', 'hybrid_context_recall',
    'combined_faithfulness', 'combined_answer_relevancy', 'combined_context_precision', 'combined_context_recall',
]

for col in metric_cols:
    if col in df.columns:
        mean_val = pd.to_numeric(df[col], errors='coerce').mean()
        print(f'  {col}: {mean_val:.4f}')

print()
print('✓ All done! finalAnalysis.csv is fully populated.')
df.head(20)

  FINAL ANALYSIS SUMMARY

Total questions: 18
Columns: 25

  ✓ question: 18/18 filled (100%)
  ✓ question_type: 18/18 filled (100%)
  ✓ ground_truth: 18/18 filled (100%)
  ✓ vectorRAG_answer: 18/18 filled (100%)
  ✓ graphRAG_answer: 18/18 filled (100%)
  ✓ hybridRAG_answer: 18/18 filled (100%)
  ✓ vectorRAG_sources: 18/18 filled (100%)
  ✓ graphRAG_triplets: 18/18 filled (100%)
  ✓ hybridRAG_sources: 18/18 filled (100%)
  ✓ vector_faithfulness: 18/18 filled (100%)
  ✓ graph_faithfulness: 18/18 filled (100%)
  ⚠ hybrid_faithfulness: 17/18 filled (94%)
  ✗ vector_answer_relevancy: 0/18 filled (0%)
  ⚠ graph_answer_relevancy: 13/18 filled (72%)
  ✓ hybrid_answer_relevancy: 18/18 filled (100%)
  ✓ vector_context_precision: 18/18 filled (100%)
  ⚠ graph_context_precision: 5/18 filled (28%)
  ✗ hybrid_context_precision: 0/18 filled (0%)
  ✓ vector_context_recall: 18/18 filled (100%)
  ✓ graph_context_recall: 18/18 filled (100%)
  ⚠ hybrid_context_recall: 16/18 filled (89%)
  ✓ combined_faith

,question,question_type,ground_truth,vectorRAG_answer,graphRAG_answer,hybridRAG_answer,vectorRAG_sources,graphRAG_triplets,hybridRAG_sources,vector_faithfulness,...,vector_context_precision,graph_context_precision,hybrid_context_precision,vector_context_recall,graph_context_recall,hybrid_context_recall,combined_faithfulness,combined_answer_relevancy,combined_context_precision,combined_context_recall
0,What is the maximum air ambulance travel dista...,Basic Retrieval,The maximum air ambulance travel distance cove...,The maximum air ambulance travel distance cove...,The maximum air ambulance travel distance cove...,The maximum air ambulance travel distance cove...,"[{""text"": ""Well Baby Well Mother- Add On Wordi...","[{""subject"": ""Air Ambulance Services"", ""subjec...","{""vector_sources"": [{""text"": ""Well Baby Well M...",0.750000,...,0.583333,1.000000,NaN,1.0,1.0,1.0,0.6250,1.0000,0.7917,1.0000
1,What expenses are included under routine medic...,Basic Retrieval,"Routine medical care includes pharmacy, diagno...","According to [Passage 2], routine medical care...",To determine the expenses included under routi...,Routine medical care for Well Mother cover inc...,"[{""text"": ""Well Baby Well Mother- Add On Wordi...","[{""subject"": ""Well Mother Cover"", ""subject_typ...","{""vector_sources"": [{""text"": ""Well Baby Well M...",1.000000,...,0.583333,0.000000,NaN,1.0,1.0,1.0,1.0000,0.8509,0.2917,1.0000
2,What preventive care services are covered for ...,Basic Retrieval,Preventive care services for a newborn baby in...,"According to the provided context, the prevent...",To determine the preventive care services cove...,Preventive care services covered for a newborn...,"[{""text"": ""Well Baby Well Mother- Add On Wordi...","[{""subject"": ""Routine preventive care services...","{""vector_sources"": [{""text"": ""Well Baby Well M...",1.000000,...,0.500000,0.000000,NaN,1.0,0.0,1.0,0.9444,1.0000,0.2500,0.6667
3,Is infertility treatment covered under the Wel...,Basic Retrieval,"No, infertility treatments are not covered und...","According to [Passage 3], infertility treatmen...",To determine if infertility treatment is cover...,"No, infertility treatment is not covered under...","[{""text"": ""Well Baby Well Mother- Add On Wordi...","[{""subject"": ""Well Mother Cover"", ""subject_typ...","{""vector_sources"": [{""text"": ""Well Baby Well M...",1.000000,...,0.333333,0.333333,NaN,1.0,1.0,1.0,0.9667,1.0000,0.3333,1.0000
4,What is the mode of payment for air ambulance ...,Basic Retrieval,Air ambulance claims are payable only through ...,The mode of payment for air ambulance claims i...,⚠️ The requested information was not found in ...,The mode of payment for air ambulance claims i...,"[{""text"": ""Well Baby Well Mother- Add On Wordi...","[{""subject"": ""Insurer"", ""subject_type"": ""INSUR...","{""vector_sources"": [{""text"": ""Well Baby Well M...",0.333333,...,0.333333,0.000000,NaN,1.0,0.0,1.0,0.6111,1.0000,0.1667,0.6667
5,If an insured person travels 300 km using an a...,Numerical Reasoning,If an insured person travels 300 km using an a...,"According to the provided context, if an insur...",To determine the reimbursement for an insured ...,If an insured person travels 300 km using an a...,"[{""text"": ""Well Baby Well Mother- Add On Wordi...","[{""subject"": ""Insured Person"", ""subject_type"":...","{""vector_sources"": [{""text"": ""Well Baby Well M...",0.714286,...,0.583333,NaN,NaN,1.0,0.0,1.0,0.6472,0.4614,0.5833,0.6667
6,What happens if the air ambulance distance exc...,Numerical Reasoning,"If the air ambulance distance exceeds 150 km, ...","If the air ambulance distance exceeds 150 km, ...",⚠️ The requested information was not found in ...,"If the air ambulance distance exceeds 150 km, ...","[{""text"": ""Well Baby Well Mother- Add On Wordi...","[{""subject"": ""Air Ambulance Services"", ""subjec...","{""vector_sources"": [{""text"": ""Well Baby Well M...",0.750000,...,0.500000,NaN,NaN,1.0,0.0,1.0,0.7917,0.5000,0.5000,0

In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv("analysis2.csv")

# Compute average metrics (NaN values are ignored automatically)
table = pd.DataFrame({
    "Metric": [
        "Faithfulness",
        "Answer Relevancy",
        "Context Precision",
        "Context Recall"
    ],
    "VectorRAG": [
        df["vector_faithfulness"].mean(),
        df["vector_answer_relevancy"].mean(),
        df["vector_context_precision"].mean(),
        df["vector_context_recall"].mean()
    ],
    "GraphRAG": [
        df["graph_faithfulness"].mean(),
        df["graph_answer_relevancy"].mean(),
        df["graph_context_precision"].mean(),
        df["graph_context_recall"].mean()
    ],
    "HybridRAG": [
        df["hybrid_faithfulness"].mean(),
        df["hybrid_answer_relevancy"].mean(),
        df["hybrid_context_precision"].mean(),
        df["hybrid_context_recall"].mean()
    ]
})
print("Analysis 2")
table


Analysis 2


,Metric,VectorRAG,GraphRAG,HybridRAG
0,Faithfulness,0.836905,0.547482,0.929412
1,Answer Relevancy,NaN,0.153846,0.839860
2,Context Precision,0.527238,0.266667,NaN
3,Context Recall,1.000000,0.333333,1.000000
